In [2]:
# CELL 1: Setup & Configuration
print("🚀 Setting up environment...")

import os
import gc
import shutil
import random
import numpy as np
import rasterio
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from google.colab import drive

# Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- HYPERPARAMETERS ---
PATCH_SIZE = 128
STRIDE = 128             # No overlap for contiguous testing
BG_KEEP_PROB = 1.0       # Keep 100% of background patches for accurate testing
MAX_NAN_RATIO = 0.10
CHUNK_SIZE = 1024

# --- PATHS ---
labels_file = Path("/content/drive/MyDrive/bow_river_wetlands_10m_final.tif")
embeddings_dir = Path("/content/drive/MyDrive/EarthEngine")
local_output_base = Path("/content/Testing_Dataset")

# --- DIRECTORY SETUP ---
if local_output_base.exists():
    shutil.rmtree(local_output_base)

dirs = {
    'images': local_output_base / 'images',
    'masks': local_output_base / 'masks'
}

for d in dirs.values():
    d.mkdir(parents=True, exist_ok=True)

# --- TILES ---
all_tiles = sorted(list(embeddings_dir.glob("*embed*.tif")))
print(f"Found {len(all_tiles)} total tiles for testing.")

# Copy labels locally for fast access
local_labels_path = Path("/content/local_labels.tif")
if not local_labels_path.exists():
    print("Copying master labels to local SSD...")
    shutil.copy(labels_file, local_labels_path)

🚀 Setting up environment...
Mounted at /content/drive
Found 88 total tiles for testing.
Copying master labels to local SSD...


In [ ]:
# CELL 2: RAM-Safe Parallel Extraction
print("\n" + "="*70)
print(f"EXTRACTING {PATCH_SIZE}x{PATCH_SIZE} TESTING PAIRS (RAM SAFE)")
print("="*70)

def extract_safely(tile_file):
    img_dir = dirs['images']
    mask_dir = dirs['masks']
    saved_count = 0
    bg_count = 0

    local_tile_path = Path("/content") / tile_file.name

    try:
        shutil.copy(tile_file, local_tile_path)

        with rasterio.open(local_tile_path) as tile_src, rasterio.open(local_labels_path) as lbl_src:
            if tile_src.count != 64:
                return 0, 0

            w, h = tile_src.width, tile_src.height

            # Read image in RAM-safe chunks
            for y_chunk in range(0, h, CHUNK_SIZE):
                for x_chunk in range(0, w, CHUNK_SIZE):

                    window_w = min(CHUNK_SIZE, w - x_chunk)
                    window_h = min(CHUNK_SIZE, h - y_chunk)

                    if window_w < PATCH_SIZE or window_h < PATCH_SIZE:
                        continue

                    window = rasterio.windows.Window(x_chunk, y_chunk, window_w, window_h)
                    chunk_img = tile_src.read(window=window)

                    chunk_bounds = rasterio.windows.bounds(window, tile_src.transform)
                    lbl_window = rasterio.windows.from_bounds(
                        *chunk_bounds, transform=lbl_src.transform
                    ).round_lengths().round_offsets()

                    chunk_mask = lbl_src.read(1, window=lbl_window)

                    # Sync dimensions
                    min_h = min(chunk_img.shape[1], chunk_mask.shape[0])
                    min_w = min(chunk_img.shape[2], chunk_mask.shape[1])
                    chunk_img = chunk_img[:, :min_h, :min_w]
                    chunk_mask = chunk_mask[:min_h, :min_w]

                    for y in range(0, min_h - PATCH_SIZE + 1, STRIDE):
                        for x in range(0, min_w - PATCH_SIZE + 1, STRIDE):

                            img_patch = chunk_img[:, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                            mask_patch = chunk_mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

                            # Do NOT skip patches with NaNs. Just fill them with 0.0 to preserve the contiguous map.
                            img_patch = np.nan_to_num(img_patch, nan=0.0).astype(np.float16)
                            mask_patch = mask_patch.astype(np.uint8)

                            # Smart Background Sampling (Will keep 100% since BG_KEEP_PROB = 1.0)
                            is_wetland = np.sum(mask_patch > 0) >= (PATCH_SIZE * PATCH_SIZE * 0.05)
                            if not is_wetland:
                                if random.random() > BG_KEEP_PROB:
                                    continue
                                bg_count += 1

                            global_y = y_chunk + y
                            global_x = x_chunk + x
                            file_id = f"{tile_file.stem}_y{global_y}_x{global_x}"

                            np.save(img_dir / f"{file_id}.npy", img_patch)
                            np.save(mask_dir / f"{file_id}.npy", mask_patch)
                            saved_count += 1

                    del chunk_img, chunk_mask

    except Exception as e:
        print(f"Error on {tile_file.name}: {e}")

    finally:
        if local_tile_path.exists():
            local_tile_path.unlink()
        gc.collect()

    return saved_count, bg_count

# Multiprocessing Execution
workers = min(os.cpu_count(), 4)

total_saved, total_bg = 0, 0
print(f"\nProcessing {len(all_tiles)} tiles using {workers} cores...")

with ProcessPoolExecutor(max_workers=workers) as executor:
    futures = {executor.submit(extract_safely, tf): tf for tf in all_tiles}
    for future in tqdm(as_completed(futures), total=len(all_tiles)):
        s, b = future.result()
        total_saved += s; total_bg += b

print(f"\n✅ Extraction Complete!")
print(f"Total testing pairs saved: {total_saved:,} (Backgrounds kept: {total_bg:,})")


EXTRACTING 128x128 TESTING PAIRS (RAM SAFE)

Processing 88 tiles using 2 cores...


 75%|███████▌  | 66/88 [1:15:13<26:30, 72.28s/it]

In [ ]:
# CELL 3: Visual Alignment Check
import matplotlib.pyplot as plt

print("\n" + "="*70)
print("VISUAL ALIGNMENT CHECK")
print("="*70)

# UPDATED: Pointing to the unified testing folders
test_images = list(dirs['images'].glob('*.npy'))

if not test_images:
    print("⚠️ No patches were saved. Check overlap or filtering logic.")
else:
    sample_img_path = random.choice(test_images)
    sample_mask_path = dirs['masks'] / sample_img_path.name

    img = np.load(sample_img_path).astype(np.float32)
    mask = np.load(sample_mask_path)

    # Take the first 3 channels for visualization
    vis_img = img[0:3, :, :].transpose(1, 2, 0)
    vis_img = (vis_img - np.nanmin(vis_img)) / (np.nanmax(vis_img) - np.nanmin(vis_img) + 1e-8)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

    ax1.imshow(vis_img)
    ax1.set_title("Embedding (First 3 Channels)")
    ax1.axis('off')

    ax2.imshow(mask, cmap='jet', interpolation='nearest')
    ax2.set_title("Wetland Label Mask")
    ax2.axis('off')

    ax3.imshow(vis_img)
    mask_overlay = np.ma.masked_where(mask == 0, mask)
    ax3.imshow(mask_overlay, cmap='autumn', alpha=0.5, interpolation='nearest')
    ax3.set_title("Overlay Alignment")
    ax3.axis('off')

    plt.tight_layout()
    plt.show()

In [6]:
# CELL 4: Compress and Transfer to Drive
print("\n" + "="*70)
print("COMPRESSING AND TRANSFERRING TO DRIVE")
print("="*70)

local_archive_base = "/content/UNet_Wetland_Dataset_Archive"
drive_destination = "/content/drive/MyDrive/Cascade_Wetland_Dataset.zip"

print("Zipping local dataset... (Fast on local SSD)")
shutil.make_archive(local_archive_base, "zip", local_output_base)

print("Moving .zip to Google Drive... (Takes a moment)")
if os.path.exists(drive_destination):
    os.remove(drive_destination)

shutil.move(f"{local_archive_base}.zip", drive_destination)

print(f"\n🎉 Success! Final v2 dataset safely stored at: {drive_destination}")


COMPRESSING AND TRANSFERRING TO DRIVE
Zipping local dataset... (Fast on local SSD)
Moving .zip to Google Drive... (Takes a moment)

🎉 Success! Final v2 dataset safely stored at: /content/drive/MyDrive/Cascade_Wetland_Dataset.zip
